In [2]:
import os, sys
sys.path.append(os.path.abspath("../.."))

import pandas as pd
import joblib
import pandas_market_calendars as mcal

from pandas.errors import EmptyDataError
from classes.trading.actionPredictionTrading import ActionPredictionTrading
from classes.neural_networks.architectures.arima_model import ArimaModel

In [3]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

In [4]:
# --- Helpers ---
b3_cal = mcal.get_calendar('B3')

def load_full_series(csv_path: str, stock: str) -> pd.Series:
    """Carrega série completa, indexada só em pregões da B3."""
    df = pd.read_csv(csv_path, parse_dates=['Date'], index_col='Date')
    df.sort_index(inplace=True)
    sched = b3_cal.schedule(start_date=df.index.min(), end_date=df.index.max())
    idx   = sched.index
    return df[stock].reindex(idx).dropna()

def rolling_forecast(model_fit, series: pd.Series, step_size: int = 1) -> pd.Series:
    """Forecast roll-forward estendendo `model_fit` com valores reais."""
    preds = []
    pos = 0
    while pos < len(series):
        h = min(step_size, len(series) - pos)
        yhat = model_fit.forecast(steps=h)
        preds.extend(yhat)
        if pos + h < len(series):
            model_fit = model_fit.extend(series.iloc[pos:pos+h].values)
        pos += h
    return pd.Series(preds, index=series.index)

In [5]:
# 

def print_detailed_performance(strategy_name: str, result_dict: dict, initial_capital: float = 100000.0):
    """
    Pega o dicionário da simulação e imprime um resumo detalhado e formatado.
    """
    # Extrai e calcula as métricas pedidas
    total_trades = result_dict.get('total_trades', 0)
    if total_trades == 0:
        print(f"\n--- Análise de Performance para '{strategy_name}' ---")
        print("    - Nenhuma operação foi realizada.")
        print("-" * 50)
        return

    hit_rate = result_dict['hit_rate']
    final_capital = result_dict['final_capital']
    max_drawdown = result_dict['max_drawdown']

    dias_de_lucro = int(total_trades * hit_rate)
    dias_de_prejuizo = total_trades - dias_de_lucro
    lucro_total_rs = final_capital - initial_capital

    print(f"\n--- Análise de Performance para '{strategy_name}' ---")
    
    print("\n  Resultado Financeiro:")
    print(f"    - Lucro/Prejuízo Total: R$ {lucro_total_rs:,.2f}")
    print(f"    - Capital Final:        R$ {final_capital:,.2f}")

    print("\n  Consistência da Estratégia:")
    print(f"    - Dias de Lucro:        {dias_de_lucro}")
    print(f"    - Dias de Prejuízo:     {dias_de_prejuizo}")
    print(f"    - Total de Trades:      {total_trades}")
    print(f"    - Taxa de Acerto (Hit Rate): {hit_rate:.2%}")

    print("\n  Análise de Risco:")
    print(f"    - Máximo Drawdown:      {max_drawdown:.2%}")
    print("-" * 50)


In [7]:
# --- Parâmetros ---
csv_path   = "../../datasets/b3_dados/processed/acoes_concat.csv"
stocks     =  [
    
    "VALE3",
    "ELET3",
]

periods    = {
    "pre_pandemia":     ("2019-01-01", "2019-12-31"),
    "durante_pandemia": ("2020-01-01", "2021-08-31"),
    "pos_pandemia":     ("2021-09-01", "2022-09-30"),
}
arima_dir  = "../../saved_models/arima"
arima_ver  = "1.0"
shares     = 100

In [11]:
# --- Backtest sem retrain ---
results = {}

for stock in stocks:
    print(f"\nAnalyzing stock: {stock}")
    # carrega ARIMA treinado
    arima_path = os.path.join(arima_dir, f"{stock}_arima_v{arima_ver}.pkl")
    arima: ArimaModel = joblib.load(arima_path)

    # carrega série completa
    series = load_full_series(csv_path, stock)

    for period_name, (start, end) in periods.items():
        print(f"\nProcessing period: {period_name} ({start} to {end})")
        subset = series[start:end]
        preds  = rolling_forecast(arima.model_fit, subset, step_size=1)

        # calcula métricas  
        mse = mean_squared_error(subset, preds)
        mae = mean_absolute_error(subset, preds)    
        rmse = np.sqrt(mse)
        r2 = r2_score(subset, preds)

        arima_metrics = {
            'mse': mse,
            'rmse': rmse,
            'mae': mae,
            'r2': r2
        }

        # monta DataFrame para o simulador
        df_bt = pd.DataFrame({
            'Date':      subset.index,
            'actual':    subset.values,
            'predicted': preds.values
        })

        # instancia para backtest
        ap = ActionPredictionTrading(df_bt, ticker='actual', window=1, model_path=None)
        # injeta a coluna de previsões
        ap.df['predicted'] = df_bt['predicted'].reset_index(drop=True)
        ap.df['actual_next'] = ap.df['actual'].shift(-1)
        # roda as simulações
        result_no_stop   = ap.simulate_trading(stop_loss=False, shares_per_trade=shares)
        result_with_stop = ap.simulate_trading(stop_loss=True,  shares_per_trade=shares)
        bh      = ap.simulate_buy_and_hold(shares=shares)

        print_detailed_performance("Estratégia SEM Stop Loss", result_no_stop)
        print_detailed_performance("Estratégia COM Stop Loss", result_with_stop)


        results[(stock, period_name)] = {
            'arima_metrics': arima_metrics,
            'arima_no_stop':   result_no_stop,
            'arima_with_stop': result_with_stop,
            'arima_bh':        bh
        }


Analyzing stock: VALE3


c:\Users\mathi\.conda\envs\tcc_project\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
c:\Users\mathi\.conda\envs\tcc_project\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
c:\Users\mathi\.conda\envs\tcc_project\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(



Processing period: pre_pandemia (2019-01-01 to 2019-12-31)

--- Análise de Performance para 'Estratégia SEM Stop Loss' ---

  Resultado Financeiro:
    - Lucro/Prejuízo Total: R$ 2,667.02
    - Capital Final:        R$ 102,667.02

  Consistência da Estratégia:
    - Dias de Lucro:        131
    - Dias de Prejuízo:     116
    - Total de Trades:      247
    - Taxa de Acerto (Hit Rate): 53.04%

  Análise de Risco:
    - Máximo Drawdown:      0.63%
--------------------------------------------------

--- Análise de Performance para 'Estratégia COM Stop Loss' ---

  Resultado Financeiro:
    - Lucro/Prejuízo Total: R$ 3,067.05
    - Capital Final:        R$ 103,067.05

  Consistência da Estratégia:
    - Dias de Lucro:        131
    - Dias de Prejuízo:     116
    - Total de Trades:      247
    - Taxa de Acerto (Hit Rate): 53.04%

  Análise de Risco:
    - Máximo Drawdown:      0.54%
--------------------------------------------------

Processing period: durante_pandemia (2020-01-01 to 

c:\Users\mathi\.conda\envs\tcc_project\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
c:\Users\mathi\.conda\envs\tcc_project\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
c:\Users\mathi\.conda\envs\tcc_project\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(



--- Análise de Performance para 'Estratégia SEM Stop Loss' ---

  Resultado Financeiro:
    - Lucro/Prejuízo Total: R$ 4,239.23
    - Capital Final:        R$ 104,239.23

  Consistência da Estratégia:
    - Dias de Lucro:        229
    - Dias de Prejuízo:     183
    - Total de Trades:      412
    - Taxa de Acerto (Hit Rate): 55.58%

  Análise de Risco:
    - Máximo Drawdown:      1.66%
--------------------------------------------------

--- Análise de Performance para 'Estratégia COM Stop Loss' ---

  Resultado Financeiro:
    - Lucro/Prejuízo Total: R$ 7,204.17
    - Capital Final:        R$ 107,204.17

  Consistência da Estratégia:
    - Dias de Lucro:        229
    - Dias de Prejuízo:     183
    - Total de Trades:      412
    - Taxa de Acerto (Hit Rate): 55.58%

  Análise de Risco:
    - Máximo Drawdown:      1.12%
--------------------------------------------------

Processing period: pos_pandemia (2021-09-01 to 2022-09-30)


...


--- Análise de Performance para 'Estratégia SEM Stop Loss' ---

  Resultado Financeiro:
    - Lucro/Prejuízo Total: R$ 425.27
    - Capital Final:        R$ 100,425.27

  Consistência da Estratégia:
    - Dias de Lucro:        122
    - Dias de Prejuízo:     125
    - Total de Trades:      247
    - Taxa de Acerto (Hit Rate): 49.39%

  Análise de Risco:
    - Máximo Drawdown:      0.99%
--------------------------------------------------

--- Análise de Performance para 'Estratégia COM Stop Loss' ---

  Resultado Financeiro:
    - Lucro/Prejuízo Total: R$ 2,281.73
    - Capital Final:        R$ 102,281.73

  Consistência da Estratégia:
    - Dias de Lucro:        122
    - Dias de Prejuízo:     125
    - Total de Trades:      247
    - Taxa de Acerto (Hit Rate): 49.39%

  Análise de Risco:
    - Máximo Drawdown:      0.41%
--------------------------------------------------

Processing period: durante_pandemia (2020-01-01 to 2021-08-31)


c:\Users\mathi\.conda\envs\tcc_project\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
c:\Users\mathi\.conda\envs\tcc_project\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
c:\Users\mathi\.conda\envs\tcc_project\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(



--- Análise de Performance para 'Estratégia SEM Stop Loss' ---

  Resultado Financeiro:
    - Lucro/Prejuízo Total: R$ 1,821.92
    - Capital Final:        R$ 101,821.92

  Consistência da Estratégia:
    - Dias de Lucro:        213
    - Dias de Prejuízo:     199
    - Total de Trades:      412
    - Taxa de Acerto (Hit Rate): 51.70%

  Análise de Risco:
    - Máximo Drawdown:      1.49%
--------------------------------------------------

--- Análise de Performance para 'Estratégia COM Stop Loss' ---

  Resultado Financeiro:
    - Lucro/Prejuízo Total: R$ 5,976.12
    - Capital Final:        R$ 105,976.12

  Consistência da Estratégia:
    - Dias de Lucro:        213
    - Dias de Prejuízo:     199
    - Total de Trades:      412
    - Taxa de Acerto (Hit Rate): 51.70%

  Análise de Risco:
    - Máximo Drawdown:      0.60%
--------------------------------------------------

Processing period: pos_pandemia (2021-09-01 to 2022-09-30)


c:\Users\mathi\.conda\envs\tcc_project\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
c:\Users\mathi\.conda\envs\tcc_project\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
c:\Users\mathi\.conda\envs\tcc_project\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(



--- Análise de Performance para 'Estratégia SEM Stop Loss' ---

  Resultado Financeiro:
    - Lucro/Prejuízo Total: R$ -425.19
    - Capital Final:        R$ 99,574.81

  Consistência da Estratégia:
    - Dias de Lucro:        132
    - Dias de Prejuízo:     137
    - Total de Trades:      269
    - Taxa de Acerto (Hit Rate): 49.07%

  Análise de Risco:
    - Máximo Drawdown:      1.50%
--------------------------------------------------

--- Análise de Performance para 'Estratégia COM Stop Loss' ---

  Resultado Financeiro:
    - Lucro/Prejuízo Total: R$ 564.08
    - Capital Final:        R$ 100,564.08

  Consistência da Estratégia:
    - Dias de Lucro:        132
    - Dias de Prejuízo:     137
    - Total de Trades:      269
    - Taxa de Acerto (Hit Rate): 49.07%

  Análise de Risco:
    - Máximo Drawdown:      1.34%
--------------------------------------------------


In [16]:

# --- flatten dos resultados em linhas de tabela ---
flat = []
for (stock, period_name), vals in results.items():
    flat.append({
        'stock':           stock,
        'period':          period_name,
        'modelo':          'ARIMA', 
        'mse':             vals['arima_metrics']['mse'],
        'rmse':            vals['arima_metrics']['rmse'],
        'mae':             vals['arima_metrics']['mae'],
        'r2':              vals['arima_metrics']['r2'],       
        'retorno_no_sl':   vals['arima_no_stop']['total_return'],
        'acerto_no_sl':    vals['arima_no_stop']['hit_rate'],
        'sharpe_no_sl':    vals['arima_no_stop']['sharpe_ratio'],
        'drawdown_no_sl':  vals['arima_no_stop']['max_drawdown'],
        'capital_no_sl':   vals['arima_no_stop']['final_capital'],
        'retorno_sl':      vals['arima_with_stop']['total_return'],
        'acerto_sl':       vals['arima_with_stop']['hit_rate'],
        'sharpe_sl':       vals['arima_with_stop']['sharpe_ratio'],
        'drawdown_sl':     vals['arima_with_stop']['max_drawdown'],
        'capital_sl':      vals['arima_with_stop']['final_capital'],
        'retorno_bh':      vals['arima_bh']['total_return'],
        'capital_bh':      vals['arima_bh']['final_capital'],
        'dias_bh':         vals['arima_bh']['days_held']
    })

df_results = pd.DataFrame(flat)

# --- caminho onde guardar ---
csv_path = "../../datasets/trading/arima_trading_results.csv"
os.makedirs(os.path.dirname(csv_path), exist_ok=True)

# --- concat incremental ---
if os.path.exists(csv_path):
    try:
        df_prev = pd.read_csv(csv_path)
    except EmptyDataError:
        # arquivo existe, mas vazio: considera DataFrame vazio
        df_prev = pd.DataFrame()
    df_comb = pd.concat([df_prev, df_results], ignore_index=True)
    df_comb.drop_duplicates(subset=['stock','period','modelo'], keep='last', inplace=True)
    df_comb.to_csv(csv_path, index=False)
    print(f"Resultados ARIMA atualizados em {csv_path}")
else:
    # não existia: cria do zero
    df_results.to_csv(csv_path, index=False)
    print(f"Arquivo ARIMA criado: {csv_path}")

# carrega para visualizar
trading_results = pd.read_csv(csv_path)
trading_results



Resultados ARIMA atualizados em ../../datasets/trading/arima_trading_results.csv


,stock,period,modelo,mse,rmse,mae,r2,retorno_no_sl,acerto_no_sl,sharpe_no_sl,drawdown_no_sl,capital_no_sl,retorno_sl,acerto_sl,sharpe_sl,drawdown_sl,capital_sl,retorno_bh,capital_bh,dias_bh
0,ITUB4,pre_pandemia,ARIMA,0.378370,0.615118,0.369044,0.722699,-0.007174,0.502024,-0.067046,0.012550,99282.564926,-0.005672,0.502024,-0.054710,0.011571,99432.837309,0.001267,100126.688004,248
1,ITUB4,durante_pandemia,ARIMA,0.509221,0.713597,0.445848,0.942738,0.013187,0.526699,0.056808,0.009053,101318.717766,0.026623,0.526699,0.129097,0.006076,102662.292755,-0.004545,99545.456886,413
2,ITUB4,pos_pandemia,ARIMA,0.274995,0.524399,0.311828,0.914480,0.009617,0.505576,0.079887,0.006263,100961.745262,0.010761,0.505576,0.091106,0.006099,101076.065855,-0.001864,99813.605690,270
3,BBAS3,pre_pandemia,ARIMA,0.253247,0.503237,0.273499,0.690639,0.005272,0.518219,0.065770,0.003402,100527.202034,0.006999,0.518219,0.091445,0.002845,100699.943577,0.002786,100278.622437,248
4,BBAS3,durante_pandemia,ARIMA,0.356937,0.597442,0.274865,0.920110,0.004424,0.500000,0.027586,0.006832,100442.394447,0.015569,0.500000,0.113927,0.004304,101556.940924,-0.007405,99259.464073,413
5,BBAS3,pos_pandemia,ARIMA,0.081645,0.285736,0.204654,0.977634,-0.000544,0.498141,-0.007657,0.008026,99945.578957,0.002358,0.498141,0.036745,0.006004,100235.808262,0.004490,100449.041367,270
6,CYRE3,pre_pandemia,ARIMA,0.160884,0.401104,0.290370,0.988487,0.000012,0.469636,0.000137,0.004519,100001.223183,0.003311,0.469636,0.039965,0.003831,100331.142767,0.011945,101194.511032,248
7,CYRE3,durante_pandemia,ARIMA,1.173980,1.083504,0.570207,0.927548,0.013381,0.519417,0.043838,0.008866,101338.087654,0.044006,0.519417,0.181827,0.003571,104400.611513,-0.006961,99303.887558,413
8,CYRE3,pos_pandemia,ARIMA,0.456579,0.675706,0.352478,0.826932,0.001206,0.475836,0.010593,0.007098,100120.612049,0.009719,0.475836,0.097668,0.004246,100971.934937,-0.001799,99820.057392,270
9,TEND3,pre_pandemia,ARIMA,0.342142,0.584929,0.369486,0.974631,0.002102,0.518219,0.018964,0.005542,100210.175323,0.006049,0.518219,0.059084,0.004457,100604.940289,0.013630,101363.044357,248


In [ ]:
trading_results["stock"].unique(), trading_results["period"].unique(), trading_results["modelo"].unique()